# 📊 Notebook 01 — Data Overview

**Goal:** Load the raw dataset, inspect its structure, data types, and quality metrics.
This is the first checkpoint of the data pipeline: understand the data before touching it.

---

## Dataset Description

The dataset contains historical video game sales data for over 16,000 titles across multiple platforms worldwide.
Each record represents one game–platform combination and includes the following columns:

| Column | Type | Description |
|---|---|---|
| `Rank` | int | Global sales rank |
| `Name` | str | Title of the game |
| `Platform` | str | Gaming platform (e.g. PS2, Wii, X360) |
| `Year` | float | Release year |
| `Genre` | str | Game genre (e.g. Action, Sports, RPG) |
| `Publisher` | str | Publisher name |
| `NA_Sales` | float | North America sales (millions) |
| `EU_Sales` | float | Europe sales (millions) |
| `JP_Sales` | float | Japan sales (millions) |
| `Other_Sales` | float | Rest-of-world sales (millions) |
| `Global_Sales` | float | Total global sales (millions) |

**Source:** Kaggle — Video Game Sales dataset (vgsales)

---

> **Pipeline context:** Raw data → **Data Overview** → Cleaning → Feature Engineering → EDA → Analysis

In [1]:
import sys
from pathlib import Path
import pandas as pd

# Ensure src/ is importable when running from notebooks/
sys.path.insert(0, str(Path.cwd().parent))

In [2]:
from src.ingestion.load_dataset import load_raw_data

df = load_raw_data()
print(f"Dataset loaded  →  {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head(10)

Shape: (16598, 11)


,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
0,1,Wii Sports,Wii,2006.0,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74
1,2,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24
2,3,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82
3,4,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00
4,5,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37
5,6,Tetris,GB,1989.0,Puzzle,Nintendo,23.20,2.26,4.22,0.58,30.26
6,7,New Super Mario Bros.,DS,2006.0,Platform,Nintendo,11.38,9.23,6.50,2.90,30.01
7,8,Wii Play,Wii,2006.0,Misc,Nintendo,14.03,9.20,2.93,2.85,29.02
8,9,New Super Mario Bros. Wii,Wii,2009.0,Platform,Nintendo,14.59,7.06,4.70,2.26,28.62
9,10,Duck Hunt,NES,1984.0,Shooter,Nintendo,26.93,0.63,0.28,0.47,28.31


In [3]:
# Column-level types and non-null counts
print("=== DataFrame Info ===")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16598 entries, 0 to 16597
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Rank          16598 non-null  int64  
 1   Name          16598 non-null  object 
 2   Platform      16598 non-null  object 
 3   Year          16327 non-null  float64
 4   Genre         16598 non-null  object 
 5   Publisher     16540 non-null  object 
 6   NA_Sales      16598 non-null  float64
 7   EU_Sales      16598 non-null  float64
 8   JP_Sales      16598 non-null  float64
 9   Other_Sales   16598 non-null  float64
 10  Global_Sales  16598 non-null  float64
dtypes: float64(6), int64(1), object(4)
memory usage: 1.4+ MB


In [4]:
# Summary statistics for all columns (numeric + categorical)
df.describe(include="all").T

,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
count,16598.000000,16598,16598,16327.000000,16598,16540,16598.000000,16598.000000,16598.000000,16598.000000,16598.000000
unique,NaN,11493,31,NaN,12,578,NaN,NaN,NaN,NaN,NaN
top,NaN,Need for Speed: Most Wanted,DS,NaN,Action,Electronic Arts,NaN,NaN,NaN,NaN,NaN
freq,NaN,12,2163,NaN,3316,1351,NaN,NaN,NaN,NaN,NaN
mean,8300.605254,NaN,NaN,2006.406443,NaN,NaN,0.264667,0.146652,0.077782,0.048063,0.537441
std,4791.853933,NaN,NaN,5.828981,NaN,NaN,0.816683,0.505351,0.309291,0.188588,1.555028
min,1.000000,NaN,NaN,1980.000000,NaN,NaN,0.000000,0.000000,0.000000,0.000000,0.010000
25%,4151.250000,NaN,NaN,2003.000000,NaN,NaN,0.000000,0.000000,0.000000,0.000000,0.060000
50%,8300.500000,NaN,NaN,2007.000000,NaN,NaN,0.080000,0.020000,0.000000,0.010000,0.170000
75%,12449.750000,NaN,NaN,2010.000000,NaN,NaN,0.240000,0.110000,0.040000,0.040000,0.470000


## 🔍 Missing Values

Understanding where data is absent before cleaning.
High missingness in a critical column (e.g. `Year` or `Publisher`) can affect downstream analysis and must be documented.

In [5]:
import matplotlib.pyplot as plt

missing = df.isnull().sum().rename("missing_count")
missing_pct = (df.isnull().mean() * 100).rename("missing_pct").round(2)
missing_summary = pd.concat([missing, missing_pct], axis=1)
print(missing_summary[missing_summary["missing_count"] > 0].to_string())

# Visualise missingness
cols_with_missing = missing[missing > 0]
if not cols_with_missing.empty:
    fig, ax = plt.subplots(figsize=(8, 3))
    cols_with_missing.sort_values().plot(kind="barh", ax=ax, color="steelblue")
    ax.set_title("Missing Value Count by Column")
    ax.set_xlabel("Number of Missing Rows")
    for bar, val in zip(ax.patches, cols_with_missing.sort_values()):
        ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height() / 2,
                f"{val:,}", va="center", fontsize=9)
    plt.tight_layout()
    plt.savefig("../reports/figures/00_missing_values.png", bbox_inches="tight")
    plt.show()
else:
    print("✅ No missing values detected.")

,missing_count,missing_pct
Rank,0,0.00
Name,0,0.00
Platform,0,0.00
Year,271,1.63
Genre,0,0.00
Publisher,58,0.35
NA_Sales,0,0.00
EU_Sales,0,0.00
JP_Sales,0,0.00
Other_Sales,0,0.00


## 📐 Unique Value Counts & Year Range

Checking cardinality of categorical columns and the temporal span of the dataset.

In [6]:
for col in ["Platform", "Genre", "Publisher"]:
    print(f"{col:<12}: {df[col].nunique():>4} unique values")

Platform: 31 unique values
Genre: 12 unique values
Publisher: 578 unique values


In [7]:
year_min = int(df["Year"].dropna().min())
year_max = int(df["Year"].dropna().max())
print(f"Year range   : {year_min} – {year_max}")
print(f"Duplicate rows: {df.duplicated().sum():,}")

Year range: 1980.0 – 2020.0
Duplicate rows: 0


## ✅ Schema Validation

We validate the raw data against a [Pandera](https://pandera.readthedocs.io/) schema that enforces expected types, value ranges, and non-null constraints.
Any failures here signal data quality issues to address in the cleaning step.

In [8]:
from src.validation.schema import validate_raw
import pandera.errors

try:
    validate_raw(df)
    print("✅ Schema validation passed — raw data conforms to the expected structure.")
except pandera.errors.SchemaErrors as exc:
    print("⚠️ Schema validation found issues:")
    print(exc.failure_cases.to_string())

Schema validation passed.


---

## 📝 Key Takeaways

- The raw dataset contains **~16,000+ game–platform records** spanning from the 1980s to the mid-2010s.
- **`Year` and `Publisher`** have the most missing values — these will be handled carefully during cleaning.
- **Genre** also has some nulls; we will fill with `"Unknown"` to retain affected rows.
- Sales columns appear healthy overall — no negatives detected. 
- Schema validation flags any raw data inconsistencies so they can be resolved before analysis.

**Next:** [`02_data_cleaning.ipynb`](02_data_cleaning.ipynb) — apply the cleaning pipeline and validate the output.